# Lab 9: Visualising and Understanding Your Model

**Before you start: go to Runtime → Change runtime type and select GPU.**

By now you have a model with results. The next question is not just 
*how accurate is it?* but *where does it fail, and why?*

Understanding your model's failures is the most direct path to improving it. 
A model that gets 80% accuracy and whose failures you understand is in a 
much better position than one that gets 85% and whose failures are opaque.

**This lab covers:**

**Core (do these):**
- Confusion matrix and per-class error analysis
- Worst-case examples — the images your model gets most wrong

**Optional (do these if time allows or if relevant to your project):**
- Grad-CAM — which image regions drive the model's decisions
- t-SNE — is the learned feature space well structured?

**Session structure:**
- First 45 minutes: work through the core sections
- Remaining time: apply these tools to your own model with TA support

**Goal for today:** leave with a written answer to: 
*what does my model get wrong, and what will I try next to fix it?*

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import zipfile, gdown
from pathlib import Path
from copy import deepcopy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load dataset and train a model to analyse
gdown.download('https://drive.google.com/uc?id=1KDMC39ba1MAL83FLLoSVSJY2KOmFR1aj', 'data.zip', quiet=True)
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
data_dir = Path('data')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(data_dir / 'train', transform=train_transform)
val_ds   = datasets.ImageFolder(data_dir / 'val',   transform=val_transform)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes: {CLASS_NAMES}')

In [ ]:
# Train a model — or load your own from a checkpoint
def train_quick(model, train_dl, val_dl, epochs=10, lr=1e-3):
    torch.manual_seed(42)
    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward()
            optimizer.step()

        model.eval()
        corr = tot = 0
        with torch.no_grad():
            for imgs, lbls in val_dl:
                imgs, lbls = imgs.to(device), lbls.to(device)
                corr += (model(imgs).argmax(1) == lbls).sum().item()
                tot  += imgs.size(0)
        print(f'Epoch {epoch:2d}/{epochs}  val_acc={corr/tot:.3f}')

    return model

backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = nn.Linear(backbone.fc.in_features, NUM_CLASSES)

print('Training model...')
model = train_quick(backbone, train_dl, val_dl, epochs=10)

## Core: collecting predictions

All visualisation techniques in this lab start from the same place: 
running the model on the validation set and collecting predictions, 
labels, probabilities, and images. We do this once and reuse throughout.

In [ ]:
def collect_predictions(model, loader):
    """
    Run the model on a dataloader and collect all predictions.
    Returns arrays of predictions, true labels, probabilities, and images.
    """
    model.eval()
    all_preds, all_labels, all_probs, all_images = [], [], [], []

    with torch.no_grad():
        for imgs, lbls in loader:
            logits = model(imgs.to(device))
            probs  = F.softmax(logits, dim=1).cpu()
            preds  = logits.argmax(1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(lbls.numpy())
            all_probs.extend(probs.numpy())
            all_images.extend(imgs)   # keep as tensors for display

    return (
        np.array(all_preds),
        np.array(all_labels),
        np.array(all_probs),
        all_images
    )


def denorm(tensor):
    """Reverse ImageNet normalisation for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return torch.clamp(tensor * std + mean, 0, 1)


print('Collecting validation predictions...')
preds, labels, probs, images = collect_predictions(model, val_dl)
accuracy = (preds == labels).mean()
print(f'Overall accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)')
print(f'Total validation examples: {len(preds)}')

---
## Section 1 (Core): Confusion Matrix and Per-Class Analysis

A confusion matrix shows which classes get confused with which. 
The diagonal is correct predictions. Off-diagonal entries are mistakes — 
and the pattern of mistakes is almost always informative.

Common patterns:
- **Symmetric confusion** between two classes suggests they look similar
- **One class predicted much more often** suggests class imbalance
- **One class always wrong** suggests a labelling issue or too few examples

In [ ]:
# Confusion matrix
cm = confusion_matrix(labels, preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Confusion matrix (counts)')

# Normalised (row-normalised = recall per class)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm, display_labels=CLASS_NAMES).plot(
    ax=axes[1], colorbar=False, cmap='Blues'
)
axes[1].set_title('Confusion matrix (normalised — diagonal = recall per class)')

plt.suptitle('Confusion matrix — rows = true class, columns = predicted class',
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class breakdown
print('Per-class performance:')
print(classification_report(labels, preds, target_names=CLASS_NAMES))

# Visualise per-class accuracy as a bar chart
per_class_acc = cm.diagonal() / cm.sum(axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASS_NAMES, per_class_acc * 100,
              color=['#2196F3' if a > 0.9 else '#FF9800' if a > 0.7 else '#F44336'
                     for a in per_class_acc])
ax.axhline(accuracy * 100, color='gray', linestyle='--',
           label=f'Overall accuracy ({accuracy*100:.1f}%)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-class accuracy')
ax.set_ylim(0, 105)
ax.legend()
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc*100:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

# Identify the weakest class
weakest_idx = np.argmin(per_class_acc)
print(f'Weakest class: {CLASS_NAMES[weakest_idx]} ({per_class_acc[weakest_idx]*100:.1f}%)')
print(f'What does it get confused with?')
confused_with = np.argsort(cm[weakest_idx])[::-1]
for idx in confused_with:
    if idx != weakest_idx:
        print(f'  {cm[weakest_idx, idx]} examples of {CLASS_NAMES[weakest_idx]} '
              f'predicted as {CLASS_NAMES[idx]}')

---
## Section 2 (Core): Worst-Case Examples

The confusion matrix tells you *what* goes wrong. 
Worst-case examples show you *what those failures look like* — 
which is usually where the insight comes from.

Inspect these images carefully:
- Are they genuinely hard (ambiguous, unusual angle, poor quality)?
- Are they mislabelled in your dataset?
- Is there a systematic pattern (all failures have similar backgrounds, 
  lighting, or pose)?

Each of these diagnoses suggests a different fix.

In [ ]:
def show_worst_cases(preds, labels, probs, images, class_names,
                     n_per_class=4):
    """
    Show the most confidently wrong predictions for each class.
    These are cases where the model was very sure but very wrong.
    """
    wrong_mask = preds != labels

    # For each true class, find the most confidently wrong examples
    n_classes = len(class_names)
    fig = plt.figure(figsize=(4 * n_per_class, 4 * n_classes))
    gs  = gridspec.GridSpec(n_classes, n_per_class, hspace=0.5, wspace=0.3)

    for cls in range(n_classes):
        # Wrong predictions for this class
        cls_wrong = np.where(wrong_mask & (labels == cls))[0]

        # Sort by confidence in the wrong prediction
        confidence = np.array([probs[i][preds[i]] for i in cls_wrong])
        sorted_idx = cls_wrong[np.argsort(confidence)[::-1]]

        for col in range(n_per_class):
            ax = fig.add_subplot(gs[cls, col])
            if col < len(sorted_idx):
                idx   = sorted_idx[col]
                img   = denorm(images[idx]).permute(1,2,0).numpy()
                true  = class_names[labels[idx]]
                pred  = class_names[preds[idx]]
                conf  = probs[idx][preds[idx]] * 100
                ax.imshow(img)
                ax.set_title(
                    f'True: {true}\nPred: {pred}\n({conf:.0f}% conf)',
                    fontsize=8, color='darkred'
                )
            else:
                ax.text(0.5, 0.5, 'No more\nerrors',
                        ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')

        # Row label
        fig.add_subplot(gs[cls, 0]).set_ylabel(
            class_names[cls], fontsize=11, fontweight='bold'
        )

    plt.suptitle(
        'Most confidently wrong predictions per class\n'
        '(sorted by confidence — most confident errors first)',
        fontsize=13
    )
    plt.show()


show_worst_cases(preds, labels, probs, images, CLASS_NAMES)

In [ ]:
def show_confusion_examples(preds, labels, probs, images,
                             class_a, class_b, class_names, n=4):
    """
    Show examples of a specific confusion pair — class_a predicted as class_b.
    Useful for understanding why two specific classes get confused.
    """
    # Find examples where true=class_a, predicted=class_b
    mask = (labels == class_a) & (preds == class_b)
    indices = np.where(mask)[0]

    if len(indices) == 0:
        print(f'No examples of {class_names[class_a]} predicted as {class_names[class_b]}')
        return

    # Sort by confidence
    confidence = np.array([probs[i][class_b] for i in indices])
    top_idx    = indices[np.argsort(confidence)[::-1]][:n]

    fig, axes = plt.subplots(1, len(top_idx), figsize=(4 * len(top_idx), 4))
    if len(top_idx) == 1:
        axes = [axes]

    for ax, idx in zip(axes, top_idx):
        img  = denorm(images[idx]).permute(1,2,0).numpy()
        conf = probs[idx][class_b] * 100
        ax.imshow(img)
        ax.set_title(
            f'True: {class_names[class_a]}\n'
            f'Pred: {class_names[class_b]}\n'
            f'({conf:.0f}% conf)',
            fontsize=9
        )
        ax.axis('off')

    plt.suptitle(
        f'Cases where {class_names[class_a]} was mistaken for {class_names[class_b]}',
        fontsize=12
    )
    plt.tight_layout()
    plt.show()


# Show the most common confusion pair
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
a, b = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
print(f'Most common mistake: {CLASS_NAMES[a]} predicted as {CLASS_NAMES[b]} '
      f'({cm_no_diag[a,b]} times)')
show_confusion_examples(preds, labels, probs, images, a, b, CLASS_NAMES)

**After inspecting the worst cases, answer these questions:**

1. Are the failures *genuinely hard* (ambiguous, unusual) or 
   are they cases a human would get right easily?
2. Is there a **systematic pattern** to the failures — 
   same background, same angle, same lighting condition?
3. Do any images look **mislabelled**?

| Diagnosis | What to try |
|---|---|
| Failures are genuinely ambiguous | Accept the error rate — or collect better data |
| Systematic pattern (e.g. all fail on dark images) | Add targeted augmentation or more diverse data |
| Mislabelled data | Review and re-label; consider label smoothing |
| One class always confused with another | Add more data for the confused class; check data collection |

---
## Section 3 (Optional): Grad-CAM

Grad-CAM answers: *which regions of the image most influenced 
this prediction?* It produces a heatmap overlaid on the image 
showing where the model was "looking".

This is most useful for catching **spurious correlations** — 
cases where the model is getting the right answer for the wrong reason 
(e.g. using the background rather than the object).

In [ ]:
# ─── OPTIONAL SECTION ───────────────────────────────────────────────
# Run this if you want to understand where your model looks.
# Particularly useful if your accuracy is high but you suspect
# the model is using spurious features.

class GradCAM:
    """
    Grad-CAM implementation using forward and backward hooks.

    Works with any CNN that has a convolutional layer before pooling.
    For ResNet, target_layer=model.layer4[-1] is a good default.
    """
    def __init__(self, model, target_layer):
        self.model         = model
        self.feature_maps  = None
        self.gradients     = None

        target_layer.register_forward_hook(self._save_features)
        target_layer.register_full_backward_hook(self._save_gradients)

    def _save_features(self, module, input, output):
        self.feature_maps = output.detach()

    def _save_gradients(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, image_tensor, class_idx=None):
        """
        Generate a Grad-CAM heatmap for image_tensor.
        image_tensor: (1, C, H, W) — single image with batch dimension
        class_idx:    which class to explain (None = predicted class)
        """
        self.model.eval()
        logits = self.model(image_tensor)

        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        self.model.zero_grad()
        logits[0, class_idx].backward()

        # Global-average-pool the gradients over spatial dims
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)

        # Weighted sum of feature maps
        cam = (weights * self.feature_maps).sum(dim=1, keepdim=True)
        cam = F.relu(cam)   # only keep positive contributions

        # Upsample to input image size
        cam = F.interpolate(
            cam, size=image_tensor.shape[2:],
            mode='bilinear', align_corners=False
        ).squeeze()

        # Normalise to [0, 1]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), class_idx


def show_gradcam(model, images, labels, probs, indices,
                 class_names, target_layer):
    """
    Show Grad-CAM heatmaps for a selection of images.
    indices: list of indices into the images/labels/probs arrays
    """
    grad_cam = GradCAM(model, target_layer)
    n        = len(indices)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = [axes]

    for row, idx in enumerate(indices):
        img_tensor = images[idx].unsqueeze(0).to(device).requires_grad_(False)
        img_tensor.requires_grad_(False)

        img_display = denorm(images[idx]).permute(1,2,0).numpy()
        true_label  = class_names[labels[idx]]
        pred_label  = class_names[probs[idx].argmax()]
        confidence  = probs[idx].max() * 100
        correct     = labels[idx] == probs[idx].argmax()

        cam, pred_cls = grad_cam.generate(img_tensor)

        # Original image
        axes[row][0].imshow(img_display)
        axes[row][0].set_title(
            f'True: {true_label}\nPred: {pred_label} ({confidence:.0f}%)\n'
            f'{'✓' if correct else '✗'}',
            color='green' if correct else 'red', fontsize=9
        )
        axes[row][0].axis('off')

        # Grad-CAM heatmap
        axes[row][1].imshow(cam, cmap='jet')
        axes[row][1].set_title('Grad-CAM heatmap\n(warm = high attention)')
        axes[row][1].axis('off')

        # Overlay
        axes[row][2].imshow(img_display)
        axes[row][2].imshow(cam, cmap='jet', alpha=0.45)
        axes[row][2].set_title('Overlay')
        axes[row][2].axis('off')

    plt.suptitle(
        'Grad-CAM: which image regions drive the prediction?\n'
        'Warm colours = regions most responsible for the predicted class.',
        fontsize=12
    )
    plt.tight_layout()
    plt.show()


# Show Grad-CAM for a mix of correct and incorrect predictions
correct_indices = np.where(preds == labels)[0][:2]
wrong_indices   = np.where(preds != labels)[0][:2]
selected        = list(correct_indices) + list(wrong_indices)

show_gradcam(
    model, images, labels, probs,
    indices=selected,
    class_names=CLASS_NAMES,
    target_layer=model.layer4[-1]   # last ResNet block
)

**What to look for in Grad-CAM:**

- **Good sign:** heatmap covers the object itself
- **Warning sign:** heatmap concentrated on background 
  (model may be using background shortcuts)
- **Warning sign:** heatmap diffuse or random for wrong predictions
- **Interesting:** compare heatmaps for correct vs wrong predictions — 
  do they look different?

---
## Section 4 (Optional): t-SNE Feature Visualisation

t-SNE reduces high-dimensional feature vectors to 2D for visualisation. 
Applied to the penultimate layer of your model, it shows whether the 
model has learned to separate classes in its internal representation.

Well-separated clusters suggest the model has learned meaningful features. 
Overlapping clusters between two specific classes explain why those 
classes get confused at prediction time.

In [ ]:
# ─── OPTIONAL SECTION ───────────────────────────────────────────────
# Run this to see whether your model's feature space is well-structured.
# Most informative when you want to understand class confusion.

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

def extract_features(model, loader, layer_name='avgpool'):
    """
    Extract intermediate feature vectors from a named layer.
    For ResNet, 'avgpool' gives 512-d vectors before the final classifier.
    """
    model.eval()
    features_list = []
    labels_list   = []

    def hook_fn(module, input, output):
        # Flatten the output (avgpool output is (batch, 512, 1, 1))
        features_list.append(output.squeeze(-1).squeeze(-1).detach().cpu())

    # Register hook on the named layer
    layer  = dict(model.named_modules())[layer_name]
    handle = layer.register_forward_hook(hook_fn)

    with torch.no_grad():
        for imgs, lbls in loader:
            model(imgs.to(device))
            labels_list.extend(lbls.numpy())

    handle.remove()

    features = torch.cat(features_list, dim=0).numpy()
    labels   = np.array(labels_list)
    return features, labels


def plot_tsne(features, labels, class_names, title='t-SNE', perplexity=30):
    """
    Compute and plot t-SNE of feature vectors.
    Uses PCA first to reduce dimensionality (standard practice for speed).
    """
    print(f'Computing t-SNE on {features.shape[0]} examples '
          f'with {features.shape[1]} features...')

    # PCA first: reduce to 50D before t-SNE (much faster)
    n_pca = min(50, features.shape[1], features.shape[0] - 1)
    pca   = PCA(n_components=n_pca)
    feats_pca = pca.fit_transform(features)
    print(f'PCA variance explained: {pca.explained_variance_ratio_.sum():.3f}')

    # t-SNE
    tsne    = TSNE(n_components=2, perplexity=perplexity,
                   random_state=42, n_iter=1000)
    feats_2d = tsne.fit_transform(feats_pca)

    # Plot
    fig, ax = plt.subplots(figsize=(9, 7))
    colors  = plt.cm.tab10(np.linspace(0, 1, len(class_names)))

    for cls_idx, (name, color) in enumerate(zip(class_names, colors)):
        mask = labels == cls_idx
        ax.scatter(
            feats_2d[mask, 0], feats_2d[mask, 1],
            c=[color], label=name, alpha=0.6, s=15
        )

    ax.legend(markerscale=2, fontsize=10)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('t-SNE dim 1')
    ax.set_ylabel('t-SNE dim 2')
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

    return feats_2d


print('Extracting features from validation set...')
features, feat_labels = extract_features(model, val_dl, layer_name='avgpool')
print(f'Feature matrix shape: {features.shape}')

feats_2d = plot_tsne(
    features, feat_labels, CLASS_NAMES,
    title='t-SNE of ResNet-18 penultimate layer (validation set)'
)

**What to look for in t-SNE:**

- **Well-separated clusters** — the model has learned discriminative 
  features for each class
- **Overlapping clusters** between two classes — explains confusion 
  between those classes in the confusion matrix
- **Outlier points** far from their class cluster — possibly 
  mislabelled or unusual examples
- **Substructure within a class** — suggests the class has distinct 
  visual subgroups the model has learned to distinguish internally

Note: t-SNE is a visualisation tool, not a metric. Do not over-interpret 
the exact positions — focus on the overall cluster structure.

---
## Your error analysis

For the rest of the session, apply these tools to your own model.

**Step 1 — Run `collect_predictions` on your validation set.**

**Step 2 — Run the confusion matrix and per-class analysis.**
Which class is weakest? What does it get confused with?

**Step 3 — Run `show_worst_cases`.**
Look at the failures carefully. Is there a systematic pattern?

**Step 4 — Write down your diagnosis.**
Use the table from Section 2 to identify the most likely cause 
and the most promising fix.

**Step 5 (Optional) — Run Grad-CAM.**
If you suspect your model is using spurious features, check where 
it is looking for correct and incorrect predictions.

**Before you leave, write answers to these questions:**
1. Which class is your model weakest on, and why?
2. Is there a systematic pattern to the failures?
3. What is the single most promising thing to try next?